# ML-04 — Search Intelligence Data Contract

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

Lane: Refresh / Content Opportunity Scoring (ranking pages by decline / opportunity).

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

**Unit of analysis:** one row = one **page** (`content_hash_id`), built by aggregating the underlying daily fact table, whose own grain is one row per (`client_hash_id`, `content_hash_id`, `report_date`).

**Table(s):** `fact_content_daily_performance` (daily fact table), from the FlyRank internship warehouse release.

**Time window (mid-panel month, avoiding the sealed `_sample`/final month):**
- **Feature window:** `2026-01-29` to `2026-02-27` (30 days) — everything a feature is allowed to see.
- **Split date:** `2026-02-28` (one-day buffer, kept out of both windows on purpose).
- **Label window:** `2026-03-01` to `2026-03-31` (31 days) — where the outcome is measured.

This keeps features strictly before the label window, per the leakage skill's timeline rule.

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

**Target / proxy:** a page is labeled `is_declining = 1` if its clicks in the label window are less than 80% of its clicks in the feature window (a 20%+ drop) — same style of threshold as `03_working_with_the_full_release.ipynb`'s `is_declining` column, adapted to my own windows.

**Features (all computed ONLY from the feature window, 2026-01-29 to 2026-02-27; confirmed against the real schema via `DESCRIBE`):**

| Feature | Aggregate | Knowable at the decision moment because... |
|---|---|---|
| `imp_total` | `SUM(gsc_impressions)`, filtered `gsc_data_available IS TRUE` | total visibility accrued strictly before the split date, excluding zero-filled non-GSC rows |
| `clk_total` | `SUM(gsc_clicks)`, filtered `gsc_data_available IS TRUE` | total traffic accrued strictly before the split date, same availability filter |
| `avg_position` | `AVG(gsc_avg_position)`, filtered `gsc_data_available IS TRUE` | a page's typical ranking position over the window; already observed, not a future event |
| `avg_engagement` | `SUM(ga4_engaged_sessions) / SUM(ga4_sessions)`, filtered `ga4_data_available IS TRUE`  | typical engagement using only real (non-zero-filled) GA4 rows, all from before the split date |
| `active_days` | `COUNT(DISTINCT report_date)` | how many days of the window actually have recorded activity; a coverage/reliability signal available before the split date |

**Context (never a model feature, only for grouping/joining):** `client_hash_id`, `content_hash_id` — pseudonyms, used only to group and split, per `flyrank-data`.

**Excluded (with reasons):**
- `trend_direction`, `trend_pct` (and anything computed in the label window) — label-derived, would leak the answer.
- Any product-decision flags (health scores, quick-win tags, needs-attention flags) — these are outputs of an existing rule, not raw observed signal; usable only as a baseline to beat, never as a feature.

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

In [1]:
%pip -q install duckdb
import os, getpass
import duckdb

HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
fact_daily = f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')"

print('Connected. Using build v20260703 daily fact table for month=2026-03 (mid-panel).')

Paste your Hugging Face READ token (hf_...): ··········
Connected. Using build v20260703 daily fact table for month=2026-03 (mid-panel).


### Query 1 Grain Check
Expect an **empty result**: no (client, content, report_date) triple should appear more than once.

In [2]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM {fact_daily}
    WHERE report_date BETWEEN DATE '2026-01-29' AND DATE '2026-03-31'
    GROUP BY client_hash_id, content_hash_id, report_date
    HAVING c > 1
    LIMIT 5
""").df()

print('Rows returned (should be 0 if grain holds):', len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows returned (should be 0 if grain holds): 0


,client_hash_id,content_hash_id,report_date,c


### Query 2 Row count + Date Span (feature window)


In [3]:
span_check = con.sql(f"""
    SELECT COUNT(*) AS row_count, MIN(report_date) AS earliest, MAX(report_date) AS latest
    FROM {fact_daily}
    WHERE report_date BETWEEN DATE '2026-01-29' AND DATE '2026-02-27'
""").df()

span_check

,row_count,earliest,latest
0,7860245,2026-01-29,2026-02-27


### Query 3 availabilty check
Rows before a client's `ga4_data_start` are zero-filled with `ga4_data_available = FALSE` — filtering avoids treating those zeros as real engagement.

In [10]:
availability_check = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        COUNT(*) FILTER (WHERE ga4_data_available IS TRUE) AS ga4_available_rows,
        COUNT(*) FILTER (WHERE gsc_data_available IS TRUE) AS gsc_available_rows
    FROM {fact_daily}
    WHERE report_date BETWEEN DATE '2026-01-29' AND DATE '2026-02-27'
""").df()

availability_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,gsc_available_rows
0,7860245,152059,2777986


## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

In [11]:
# 4a. Build the honest feature frame (feature window only)
features = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_impressions END) AS imp_total,
        SUM(CASE WHEN gsc_data_available IS TRUE THEN gsc_clicks END) AS clk_total,
        AVG(CASE WHEN gsc_data_available IS TRUE THEN gsc_avg_position END) AS avg_position,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_engaged_sessions END) * 1.0
            / NULLIF(SUM(CASE WHEN ga4_data_available IS TRUE THEN ga4_sessions END), 0) AS avg_engagement,
        COUNT(DISTINCT report_date) AS active_days
    FROM {fact_daily}
    WHERE report_date BETWEEN DATE '2026-01-29' AND DATE '2026-02-27'
    GROUP BY client_hash_id, content_hash_id
""").df()

print(f'{len(features):,} pages with feature-window data')
features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

317,503 pages with feature-window data


,client_hash_id,content_hash_id,imp_total,clk_total,avg_position,avg_engagement,active_days
0,client_e547b89c05043229,content_1f9a60824bdd44ea,390.0,1.0,12.052638,0.0,30
1,client_e547b89c05043229,content_0cc6d047d552023f,4039.0,10.0,5.492219,0.0,30
2,client_e547b89c05043229,content_b962dd8115b75719,549.0,1.0,3.109817,NaN,30
3,client_e547b89c05043229,content_d606939ec54831d3,526.0,1.0,15.209547,0.0,30
4,client_e547b89c05043229,content_c06834d9f426a3d6,115.0,0.0,20.540585,0.0,30


In [12]:
# 4b Build the label from label window
label_window = con.sql(f"""
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS clk_last30,
        SUM(gsc_clicks) * 100.0 / NULLIF(31, 0) AS clk_last30_daily_avg
    FROM {fact_daily}
    WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-31'
    GROUP BY client_hash_id, content_hash_id
""").df()

data = features.merge(label_window, on=['client_hash_id', 'content_hash_id'], how='inner')
data = data[data['clk_total'] > 0].copy()  # avoid divide-by-zero pages

data['pct_change'] = (data['clk_last30'] - data['clk_total']) / data['clk_total'] * 100
data['is_declining'] = (data['clk_last30'] < 0.8 * data['clk_total']).astype(int)

print(f'{len(data):,} pages with both windows | declining rate: {data["is_declining"].mean():.3f}')
data[['imp_total','clk_total','avg_position','avg_engagement','active_days','clk_last30','pct_change','is_declining']].head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

54,158 pages with both windows | declining rate: 0.431


,imp_total,clk_total,avg_position,avg_engagement,active_days,clk_last30,pct_change,is_declining
0,390.0,1.0,12.052638,0.0,30,1.0,0.0,0
1,4039.0,10.0,5.492219,0.0,30,16.0,60.0,0
2,549.0,1.0,3.109817,NaN,30,1.0,0.0,0
3,526.0,1.0,15.209547,0.0,30,1.0,0.0,0
7,547.0,1.0,2.536466,0.0,30,1.0,0.0,0


In [13]:
# 4c Honest Score (5 features only)
import numpy as np
from sklearn.tree import DecisionTreeClassifier, export_text

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

honest_features = ['imp_total', 'clk_total', 'avg_position', 'avg_engagement', 'active_days']
X_honest = data[honest_features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = data['is_declining'].values

tree_honest = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42)
tree_honest.fit(X_honest, y)
honest_scores = tree_honest.predict_proba(X_honest)[:, 1]

print('Base rate (declining share):', round(y.mean(), 3))
print('Honest Precision@50:', round(precision_at_k(honest_scores, y, 50), 3))
print(export_text(tree_honest, feature_names=honest_features))

Base rate (declining share): 0.431
Honest Precision@50: 0.66
|--- imp_total <= 456.50
|   |--- active_days <= 9.50
|   |   |--- class: 0
|   |--- active_days >  9.50
|   |   |--- class: 1
|--- imp_total >  456.50
|   |--- active_days <= 23.50
|   |   |--- class: 0
|   |--- active_days >  23.50
|   |   |--- class: 0



In [14]:
# 4d The trap - add pct-change
leaky_features = honest_features + ['pct_change']
X_leaky = data[leaky_features].replace([np.inf, -np.inf], np.nan).fillna(0)

tree_leaky = DecisionTreeClassifier(max_depth=2, class_weight='balanced', random_state=42)
tree_leaky.fit(X_leaky, y)
leaky_scores = tree_leaky.predict_proba(X_leaky)[:, 1]

print('Leaky Precision@50:', round(precision_at_k(leaky_scores, y, 50), 3), ' <- looks amazing')
print(export_text(tree_leaky, feature_names=leaky_features))

Leaky Precision@50: 1.0  <- looks amazing
|--- pct_change <= -20.09
|   |--- class: 1
|--- pct_change >  -20.09
|   |--- avg_engagement <= 0.00
|   |   |--- class: 0
|   |--- avg_engagement >  0.00
|   |   |--- class: 0



### 4e. Remove the leak, keep the honest number

**Finding:** adding `pct_change` pushed Precision@50 close to 1.0 because `pct_change` is mathematically the same comparison the label (`is_declining`) is thresholded from (label window clicks vs. feature window clicks, at a 20% cutoff) — the tree just rediscovers that threshold instantly. This is not a stronger model; it is the answer sheet in disguise. `pct_change` is removed from the feature set. The honest, trustworthy number is the **Precision@50 from section 4c**, not 4d.

## Named limitation

Because client history is an unbalanced panel, clients whose `gsc_data_start`/`ga4_data_start` fall inside or after the feature window contribute fewer rows (or none) — so row counts below the theoretical max (all clients × all pages × 30 days) are expected, not a bug. For the same reason, `avg_engagement` is GSC-only (unmeasured, not zero) for any page whose client's GA4 history hadn't started yet during the feature window, so this analysis can't say anything about engagement for that slice, and engagement-driven comparisons aren't fair across clients with different GA4 start dates.

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.